In [1]:
# For text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora
from gensim.models import LdaModel
import pandas as pd

# Download NLTK Resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
# Load the Data
df = pd.read_csv('news_dataset.csv')
df = df[['text']].dropna()
documents = df['text'].tolist()

In [6]:
stop_words = set(stopwords.words('english')) # Create a set of English stopwords
lemmatizer = WordNetLemmatizer() # Initialize a WordNet lemmatizer

def preprocess_text(text):
    tokens = word_tokenize(text.lower()) # Tokenize the text into words and convert to lowercase
    tokens = [token for token in tokens if token.isalnum()] # Filter out non-alphanumeric tokens
    tokens = [token for token in tokens if token not in stop_words] # Remove stopwords from the tokens
    tokens = [lemmatizer.lemmatize(token) for token in tokens] # Lemmatize each token
    return tokens # Return the preprocessed tokens
    
preprocessed_documents = [preprocess_text(doc) for doc in documents] # Preprocess each document in the list

print(preprocessed_documents[0])

['wondering', 'anyone', 'could', 'enlighten', 'car', 'saw', 'day', 'sport', 'car', 'looked', 'late', 'early', '70', 'called', 'bricklin', 'door', 'really', 'small', 'addition', 'front', 'bumper', 'separate', 'rest', 'body', 'know', 'anyone', 'tellme', 'model', 'name', 'engine', 'spec', 'year', 'production', 'car', 'made', 'history', 'whatever', 'info', 'funky', 'looking', 'car', 'please']


In [8]:
# Create a Gensim Dictionary object from the preprocessed documents
dictionary = corpora.Dictionary(preprocessed_documents)

# Filter out tokens that appear in less than 15 documents or more than 50% of the documents
dictionary.filter_extremes(no_below=15, no_above=0.5)

# Convert each preprocessed document into a bag-of-words representation using the dictionary
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

In [10]:
# Run LDA
lda_model = LdaModel(corpus, num_topics=5, id2word=dictionary, passes=15) # Train an LDA model on the corpus with 2 topics using Gensim's LdaModel class

In [12]:
from gensim.models import CoherenceModel
# Evaluate the LDA model using Coherence Score
coherence_model_lda = CoherenceModel(model=lda_model, texts=preprocessed_documents, dictionary=dictionary, coherence='c_v')
coherence_score = coherence_model_lda.get_coherence()

print("Coherence Score:", coherence_score)
print()

Coherence Score: 0.6610706194524678



In [13]:
# empty list to store dominant topic labels for each document
article_labels = []

# iterate over each processed document
for i, doc in enumerate(preprocessed_documents):
    # for each document, convert to bag-of-words representation
    bow = dictionary.doc2bow(doc)
    # get list of topic probabilities
    topics = lda_model.get_document_topics(bow)
    # determine topic with highest probability
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    # append to the list
    article_labels.append(dominant_topic)
    
# Create DataFrame
df_result = pd.DataFrame({"Article": documents, "Topic": article_labels})

# Print the DataFrame
print("Table with Articles and Topic:")
print(df_result)
print()

Table with Articles and Topic:
                                                 Article  Topic
0      I was wondering if anyone out there could enli...      3
1      I recently posted an article asking what kind ...      3
2      \nIt depends on your priorities.  A lot of peo...      3
3      an excellent automatic can be found in the sub...      4
4      : Ford and his automobile.  I need information...      3
...                                                  ...    ...
11091  Secrecy in Clipper Chip\n\nThe serial number o...      4
11092  Hi !\n\nI am interested in the source of FEAL ...      4
11093  The actual algorithm is classified, however, t...      4
11094  \n\tThis appears to be generic calling upon th...      1
11095  \nProbably keep quiet and take it, lest they g...      3

[11096 rows x 2 columns]



In [14]:
# Print top terms for each topic
for topic_id in range(lda_model.num_topics):
    print(f"Top terms for Topic #{topic_id}:")
    top_terms = lda_model.show_topic(topic_id, topn=10)
    print([term[0] for term in top_terms])
    print()

Top terms for Topic #0:
['x', 'db', 'file', 'program', 'information', 'encryption', 'system', 'available', 'use', 'privacy']

Top terms for Topic #1:
['people', 'would', 'one', 'think', 'say', 'government', 'know', 'god', 'right', 'u']

Top terms for Topic #2:
['1', 'q', '0', 'max', '2', '7', 'g', 'r', '3', 'p']

Top terms for Topic #3:
['year', 'game', 'would', 'get', 'one', 'team', 'like', 'time', 'go', 'think']

Top terms for Topic #4:
['key', 'chip', 'use', 'one', 'would', 'get', 'window', 'system', 'like', 'know']



In [15]:
# Print the top terms for each topic with weight
print("Top Terms for Each Topic:")
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        weight, word = term.split("*")
        print(f"- {word.strip()} (weight: {weight.strip()})")
    print()

Top Terms for Each Topic:
Topic 0:
- "x" (weight: 0.028)
- "db" (weight: 0.009)
- "file" (weight: 0.009)
- "program" (weight: 0.009)
- "information" (weight: 0.008)
- "encryption" (weight: 0.007)
- "system" (weight: 0.006)
- "available" (weight: 0.006)
- "use" (weight: 0.006)
- "privacy" (weight: 0.005)

Topic 1:
- "people" (weight: 0.011)
- "would" (weight: 0.010)
- "one" (weight: 0.009)
- "think" (weight: 0.005)
- "say" (weight: 0.005)
- "government" (weight: 0.005)
- "know" (weight: 0.005)
- "god" (weight: 0.005)
- "right" (weight: 0.005)
- "u" (weight: 0.005)

Topic 2:
- "1" (weight: 0.062)
- "q" (weight: 0.051)
- "0" (weight: 0.047)
- "max" (weight: 0.044)
- "2" (weight: 0.042)
- "7" (weight: 0.029)
- "g" (weight: 0.029)
- "r" (weight: 0.028)
- "3" (weight: 0.026)
- "p" (weight: 0.024)

Topic 3:
- "year" (weight: 0.010)
- "game" (weight: 0.009)
- "would" (weight: 0.008)
- "get" (weight: 0.008)
- "one" (weight: 0.007)
- "team" (weight: 0.007)
- "like" (weight: 0.006)
- "time" (weig